# SC4020 Project 2 — Task 1: Analysis of Drug–Target Datasets

## Step 1: Load the data and check it

This notebook expects a `data/` folder next to it, containing `davis/` and `kiba/` from the DeepDTA repository.

In [ ]:
# Run once to install the libraries (skip if already installed)
%pip install pandas numpy matplotlib seaborn

In [ ]:
import json
import pickle
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")

### Load ligands, targets and the score matrix

`ligands_can.txt` and `proteins.txt` are JSON dictionaries (ID → SMILES / sequence). `Y` is a pickled matrix whose rows follow the ligand order and columns follow the target order, so the dictionary order must be kept.

In [ ]:
def load_dataset(name):
    folder = DATA_DIR / name
    with open(folder / "ligands_can.txt") as f:
        ligands = json.load(f, object_pairs_hook=OrderedDict)
    with open(folder / "proteins.txt") as f:
        proteins = json.load(f, object_pairs_hook=OrderedDict)
    with open(folder / "Y", "rb") as f:
        Y = np.asarray(pickle.load(f, encoding="latin1"), dtype=float)  # file was saved with Python 2
    return ligands, proteins, Y

davis_ligands, davis_proteins, davis_Y = load_dataset("davis")
kiba_ligands, kiba_proteins, kiba_Y = load_dataset("kiba")

### Sanity checks against the project brief

Expected: DAVIS 68 × 442 with 30,056 observed pairs; KIBA 2,111 × 229 with 118,254 observed pairs.

In [ ]:
def summarise(name, ligands, proteins, Y):
    observed = np.isfinite(Y).sum()
    return {
        "dataset": name,
        "ligands": len(ligands),
        "targets": len(proteins),
        "matrix shape": Y.shape,
        "observed pairs": int(observed),
        "missing pairs": int(Y.size - observed),
        "sparsity": round(1 - observed / Y.size, 4),
        "unique SMILES": len(set(ligands.values())),
        "unique sequences": len(set(proteins.values())),
        "min score": np.nanmin(Y),
        "max score": np.nanmax(Y),
    }

pd.DataFrame([
    summarise("DAVIS (Kd, nM)", davis_ligands, davis_proteins, davis_Y),
    summarise("KIBA", kiba_ligands, kiba_proteins, kiba_Y),
]).set_index("dataset").T

### Example records (long format)

One row per observed ligand–target pair, which is the format the later tasks will use.

In [ ]:
def to_long(ligands, proteins, Y):
    lig_ids, lig_smiles = list(ligands.keys()), list(ligands.values())
    tgt_ids, tgt_seqs = list(proteins.keys()), list(proteins.values())
    rows, cols = np.where(np.isfinite(Y))
    return pd.DataFrame({
        "ligand_id": [lig_ids[i] for i in rows],
        "target_id": [tgt_ids[j] for j in cols],
        "smiles": [lig_smiles[i] for i in rows],
        "sequence": [tgt_seqs[j] for j in cols],
        "score": Y[rows, cols],
    })

davis_df = to_long(davis_ligands, davis_proteins, davis_Y)
kiba_df = to_long(kiba_ligands, kiba_proteins, kiba_Y)

def preview(df, n=5):
    # shorten long strings so the table is readable
    out = df.head(n).copy()
    out["smiles"] = out["smiles"].str.slice(0, 30) + "…"
    out["sequence"] = out["sequence"].str.slice(0, 30) + "…"
    return out

display(preview(davis_df))
display(preview(kiba_df))